In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler


In [2]:
BASE = Path("../data/processed/primary")

INPUT_FILE = BASE / "outpatient_ml_preprocessed.csv"
SCORE_FILE = BASE / "outpatient_anomaly_scores.csv"


In [3]:
CHUNK_SIZE = 250_000
TRAIN_SAMPLE_SIZE = 150_000
RANDOM_STATE = 42

In [4]:

header = pd.read_csv(INPUT_FILE, nrows=0)

FEATURE_COLS = list(header.columns)

print("=" * 80)
print("OUTPATIENT ANOMALY DETECTION")
print("=" * 80)

print("Feature count:", len(FEATURE_COLS))
print("Training sample:", f"{TRAIN_SAMPLE_SIZE:,}")
print("Chunk size:", f"{CHUNK_SIZE:,}")


OUTPATIENT ANOMALY DETECTION
Feature count: 51
Training sample: 150,000
Chunk size: 250,000


In [5]:

# STEP 1 — BUILD REPRESENTATIVE TRAINING SAMPLE

print("\nCreating training sample...")

sample_parts = []
sample_rows = 0

for chunk in pd.read_csv(
    INPUT_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    remaining = TRAIN_SAMPLE_SIZE - sample_rows

    if remaining <= 0:
        break

    take = min(
        remaining,
        len(chunk)
    )

    sample = chunk.sample(
        n=take,
        random_state=RANDOM_STATE
    )

    sample_parts.append(sample)
    sample_rows += len(sample)

    print(
        f"Collected {sample_rows:,} "
        f"/ {TRAIN_SAMPLE_SIZE:,}"
    )

    if sample_rows >= TRAIN_SAMPLE_SIZE:
        break

train_df = pd.concat(
    sample_parts,
    ignore_index=True
)

del sample_parts

print("\nTraining sample shape:")
print(train_df.shape)




Creating training sample...
Collected 150,000 / 150,000

Training sample shape:
(150000, 51)


In [6]:
# STEP 2 — ROBUST SCALING

print("\nScaling features...")

scaler = RobustScaler()

X_train = scaler.fit_transform(
    train_df[FEATURE_COLS]
)

X_train = np.asarray(
    X_train,
    dtype=np.float32
)

del train_df

print(
    "Scaled training matrix:",
    X_train.shape
)


Scaling features...
Scaled training matrix: (150000, 51)


In [7]:

# STEP 3 — TRAIN ISOLATION FOREST

print("\nTraining Isolation Forest...")

model = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train)

del X_train

print("Model training complete.")



Training Isolation Forest...
Model training complete.


In [8]:

# STEP 4 — SCORE ALL OUTPATIENT CLAIMS
print("\nScoring all Outpatient claims...")

if SCORE_FILE.exists():
    SCORE_FILE.unlink()

first_chunk = True
total_scored = 0

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    X = chunk[FEATURE_COLS]

    X_scaled = scaler.transform(X)

    X_scaled = np.asarray(
        X_scaled,
        dtype=np.float32
    )

    # Higher score = more anomalous
    anomaly_score = -model.decision_function(
        X_scaled
    )

    result = pd.DataFrame({
        "ANOMALY_SCORE": anomaly_score
    })

    result.to_csv(
        SCORE_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    total_scored += len(chunk)

    del X
    del X_scaled
    del result

    print(
        f"Scored {total_scored:,} claims..."
    )

print("\n" + "=" * 80)
print("OUTPATIENT MODEL COMPLETE")
print("=" * 80)

print(
    "Claims scored:",
    f"{total_scored:,}"
)

print(
    "Expected:",
    f"{790_790:,}"
)

print(
    "Score file:",
    SCORE_FILE
)


Scoring all Outpatient claims...
Scored 250,000 claims...
Scored 500,000 claims...
Scored 750,000 claims...
Scored 790,790 claims...

OUTPATIENT MODEL COMPLETE
Claims scored: 790,790
Expected: 790,790
Score file: ..\data\processed\primary\outpatient_anomaly_scores.csv


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")
SCORE_FILE = BASE / "outpatient_anomaly_scores.csv"

scores = pd.read_csv(SCORE_FILE)

print("=" * 90)
print("OUTPATIENT ANOMALY SCORE ANALYSIS")
print("=" * 90)

print("\nScore rows:", f"{len(scores):,}")
print("Score columns:", list(scores.columns))

print("\nMissing scores:", scores["ANOMALY_SCORE"].isna().sum())
print(
    "Infinite scores:",
    np.isinf(scores["ANOMALY_SCORE"]).sum()
)
print(
    "Unique scores:",
    scores["ANOMALY_SCORE"].nunique()
)

print("\n" + "=" * 90)
print("ANOMALY SCORE DISTRIBUTION")
print("=" * 90)

print(
    scores["ANOMALY_SCORE"].describe(
        percentiles=[
            0.001,
            0.005,
            0.01,
            0.02,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

print("\n" + "=" * 90)
print("HIGH ANOMALY THRESHOLDS")
print("=" * 90)

for percentile in [
    90,
    95,
    97,
    98,
    99,
    99.5,
    99.9
]:

    threshold = np.percentile(
        scores["ANOMALY_SCORE"],
        percentile
    )

    count = (
        scores["ANOMALY_SCORE"] >= threshold
    ).sum()

    print(
        f"Top {100-percentile:.1f}% | "
        f"threshold={threshold:.6f} | "
        f"claims={count:,}"
    )

print("\n" + "=" * 90)
print("SCORE RANGE")
print("=" * 90)

print("Minimum:", scores["ANOMALY_SCORE"].min())
print("Maximum:", scores["ANOMALY_SCORE"].max())
print("Mean:", scores["ANOMALY_SCORE"].mean())
print("Median:", scores["ANOMALY_SCORE"].median())

print("\n" + "=" * 90)
print("TOP 20 ANOMALY SCORES")
print("=" * 90)

print(
    scores
    .nlargest(20, "ANOMALY_SCORE")
    .to_string(index=False)
)

OUTPATIENT ANOMALY SCORE ANALYSIS

Score rows: 790,790
Score columns: ['ANOMALY_SCORE']

Missing scores: 0
Infinite scores: 0
Unique scores: 779621

ANOMALY SCORE DISTRIBUTION
count    790790.000000
mean         -0.058499
std           0.044459
min          -0.137247
0.1%         -0.129423
0.5%         -0.124010
1%           -0.120960
2%           -0.117131
5%           -0.110584
10%          -0.103769
25%          -0.089613
50%          -0.067834
75%          -0.038910
90%          -0.003241
95%           0.028687
99%           0.100778
99.5%         0.118775
99.9%         0.137560
max           0.200365
Name: ANOMALY_SCORE, dtype: float64

HIGH ANOMALY THRESHOLDS
Top 10.0% | threshold=-0.003241 | claims=79,079
Top 5.0% | threshold=0.028687 | claims=39,540
Top 3.0% | threshold=0.057383 | claims=23,724
Top 2.0% | threshold=0.077283 | claims=15,816
Top 1.0% | threshold=0.100778 | claims=7,908
Top 0.5% | threshold=0.118775 | claims=3,956
Top 0.1% | threshold=0.137560 | claims=806

SCORE 

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

SCORE_FILE = BASE / "outpatient_anomaly_scores.csv"

# Use the original processed outpatient claims
DATA_FILE = BASE / "Outpatient_Claims_Processed.csv"

OUTPUT_FILE = BASE / "outpatient_top_anomalies.csv"

TOP_N = 100
CHUNK_SIZE = 250_000

# ============================================================
# LOAD SCORES
# ============================================================

scores = pd.read_csv(SCORE_FILE)

scores["ROW_ID"] = np.arange(len(scores))

top_rows = scores.nlargest(
    TOP_N,
    "ANOMALY_SCORE"
)[["ROW_ID", "ANOMALY_SCORE"]]

top_indices = set(top_rows["ROW_ID"])

print("=" * 80)
print("RETRIEVING TOP OUTPATIENT ANOMALIES")
print("=" * 80)

print("Scores:", f"{len(scores):,}")
print("Top anomalies:", TOP_N)

# ============================================================
# FIRST CHECK THE ACTUAL COLUMNS
# ============================================================

header = pd.read_csv(DATA_FILE, nrows=0)

DETAIL_COLS = [
    "CLAIM_KEY",
    "DESYNPUF_ID",
    "CLM_ID",
    "PRVDR_NUM",
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "CLAIM_DURATION_DAYS",
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "CLAIM_YEAR",
    "CLM_FROM_DT",
    "CLM_THRU_DT",
]

available = set(header.columns)

DETAIL_COLS = [
    c for c in DETAIL_COLS
    if c in available
]

print("\nColumns being retrieved:")
print(DETAIL_COLS)

# ============================================================
# STREAM DATA
# ============================================================

matches = []
current_row = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_FILE,
        usecols=DETAIL_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    start = current_row
    end = current_row + len(chunk)

    relevant = [
        idx for idx in top_indices
        if start <= idx < end
    ]

    if relevant:

        local_indices = [
            idx - start
            for idx in relevant
        ]

        selected = chunk.iloc[
            local_indices
        ].copy()

        selected["ROW_ID"] = relevant

        matches.append(selected)

    current_row = end

    print(
        f"Chunk {chunk_number}: "
        f"{current_row:,} rows processed"
    )

# ============================================================
# COMBINE
# ============================================================

claims = pd.concat(
    matches,
    ignore_index=True
)

claims = claims.merge(
    top_rows,
    on="ROW_ID",
    how="left"
)

claims = claims.sort_values(
    "ANOMALY_SCORE",
    ascending=False
).reset_index(drop=True)

claims["ANOMALY_RANK"] = (
    np.arange(len(claims)) + 1
)

# Put ranking columns first

first_cols = [
    "ANOMALY_RANK",
    "ANOMALY_SCORE",
]

first_cols += [
    c for c in [
        "CLAIM_KEY",
        "CLM_ID",
        "DESYNPUF_ID",
        "PRVDR_NUM"
    ]
    if c in claims.columns
]

remaining_cols = [
    c for c in claims.columns
    if c not in first_cols + ["ROW_ID"]
]

claims = claims[
    first_cols + remaining_cols
]

# ============================================================
# SAVE
# ============================================================

claims.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 80)
print("TOP OUTPATIENT ANOMALIES SAVED")
print("=" * 80)

print("Rows:", len(claims))
print("Saved:", OUTPUT_FILE)

print("\nTOP 20:")
print(
    claims.head(20).to_string(index=False)
)

RETRIEVING TOP OUTPATIENT ANOMALIES
Scores: 790,790
Top anomalies: 100

Columns being retrieved:
['DESYNPUF_ID', 'CLM_ID', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'CLM_FROM_DT', 'CLM_THRU_DT']
Chunk 1: 250,000 rows processed
Chunk 2: 500,000 rows processed
Chunk 3: 750,000 rows processed
Chunk 4: 790,790 rows processed

TOP OUTPATIENT ANOMALIES SAVED
Rows: 100
Saved: ..\data\processed\primary\outpatient_top_anomalies.csv

TOP 20:
 ANOMALY_RANK  ANOMALY_SCORE          CLM_ID      DESYNPUF_ID PRVDR_NUM  CLM_FROM_DT  CLM_THRU_DT  CLM_PMT_AMT  NCH_PRMRY_PYR_CLM_PD_AMT
            1       0.200365 542852281547374 7C3AA9C0A4A8CB96    4700HR   20081201.0   20081221.0       1800.0                    5000.0
            2       0.189688 542902281474162 7C3AA9C0A4A8CB96    33016V   20081211.0   20081231.0       2700.0                      90.0
            3       0.182061 542632281494557 000B4662348C35B4    4600BC   20080304.0   20080324.0       1900.0                    5000.0
  

In [12]:
print("=" * 80)
print("CHECKING TRAINED ANOMALY MODELS")
print("=" * 80)

for name in [
    "carrier_model",
    "outpatient_model",
    "inpatient_model",
    "carrier_if",
    "outpatient_if",
    "inpatient_if",
    "carrier_scaler",
    "outpatient_scaler",
    "inpatient_scaler",
    "scaler",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

print("=" * 80)

CHECKING TRAINED ANOMALY MODELS
scaler: RobustScaler


In [13]:
from sklearn.ensemble import IsolationForest

print("=" * 80)
print("TRAINED ISOLATION FOREST OBJECTS")
print("=" * 80)

for name, obj in globals().items():
    try:
        if isinstance(obj, IsolationForest):
            print(f"{name} -> IsolationForest")
    except:
        pass

print("\n" + "=" * 80)
print("SCALERS")
print("=" * 80)

from sklearn.preprocessing import RobustScaler, StandardScaler

for name, obj in globals().items():
    try:
        if isinstance(obj, (RobustScaler, StandardScaler)):
            print(f"{name} -> {type(obj).__name__}")
    except:
        pass

TRAINED ISOLATION FOREST OBJECTS
model -> IsolationForest

SCALERS
scaler -> RobustScaler


In [14]:
import joblib
import json
from pathlib import Path

# ============================================================
# SAVE OUTPATIENT ANOMALY MODEL
# ============================================================

MODEL_DIR = Path("../models/anomaly/outpatient")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    model,
    MODEL_DIR / "isolation_forest.joblib"
)

joblib.dump(
    scaler,
    MODEL_DIR / "scaler.joblib"
)

feature_schema = {
    "claim_type": "OUTPATIENT",
    "model_type": "IsolationForest",
    "scaler_type": type(scaler).__name__,
    "feature_count": len(FEATURE_COLS),
    "features": list(FEATURE_COLS)
}

with open(
    MODEL_DIR / "feature_schema.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(feature_schema, f, indent=2)

print("=" * 80)
print("OUTPATIENT MODEL SAVED")
print("=" * 80)

print("Model:", MODEL_DIR / "isolation_forest.joblib")
print("Scaler:", MODEL_DIR / "scaler.joblib")
print("Schema:", MODEL_DIR / "feature_schema.json")
print("Features:", len(FEATURE_COLS))
print("Scaler type:", type(scaler).__name__)

OUTPATIENT MODEL SAVED
Model: ..\models\anomaly\outpatient\isolation_forest.joblib
Scaler: ..\models\anomaly\outpatient\scaler.joblib
Schema: ..\models\anomaly\outpatient\feature_schema.json
Features: 51
Scaler type: RobustScaler
